In [1]:
"hello world"

'hello world'

In [6]:
from typing import TypedDict,Annotated

from langchain_openai import ChatOpenAI
from langchain_core.messages import AnyMessage,AIMessage,BaseMessage

from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt,Command

from dotenv import load_dotenv
load_dotenv()

True

In [8]:
llm = ChatOpenAI(
    model="qwen/qwen3.8-max:free",
    base_url="https://api.xkiro.com/v1"
)

In [9]:
llm.invoke("hello")

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 152, 'total_tokens': 161, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen/qwen3.8-max:free', 'system_fingerprint': None, 'id': 'chatcmpl-5f0725ec-2cdf-465f-b0a8-da1a080eae83', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0c3f8-f0e4-71c3-9c27-7160362bf87e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 152, 'output_tokens': 9, 'total_tokens': 161, 'input_token_details': {}, 'output_token_details': {}})

In [10]:
class ChatState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]

In [11]:
def chat_node(state:ChatState):
    decision=interrupt({
        "type":"approval",
        "reason":"model is about to answer a user question",
        "question":state["messages"][-1].content,
        "instuction":"approve this question? yes/no"
    })
    if decision["approved"]=="no":
        return {"messages":[AIMessage(content="not approved.")]}
    else:
        response=llm.invoke(state["messages"])
        return {"messages":[response]}


In [23]:
builder=StateGraph(ChatState)

builder.add_node("chat_node",chat_node)

builder.add_edge(START,"chat_node")
builder.add_edge("chat_node",END)

checkpointer=MemorySaver()

app=builder.compile(checkpointer=checkpointer)


In [24]:
config={"configurable":{"thread_id":"1234"}}

initial_input={
    "messages":[
        ("user","explain dl(deep learning) in very simple terms.")
    ]
}

result=app.invoke(initial_input,config=config)

In [25]:
result

{'messages': [HumanMessage(content='explain dl(deep learning) in very simple terms.', additional_kwargs={}, response_metadata={}, id='0a336082-5570-4999-b061-a9b4d15bf1fb')],
 '__interrupt__': [Interrupt(value={'type': 'approval', 'reason': 'model is about to answer a user question', 'question': 'explain dl(deep learning) in very simple terms.', 'instuction': 'approve this question? yes/no'}, id='1066bd6741c0d9395c1a5045b7b63dcd')]}

In [26]:
user_input=input("approve this question? yes/no")

In [27]:
final_input=app.invoke(
    Command(resume={"approved":user_input}),
    config=config
)

In [28]:
final_input["messages"][1].content

'not approved.'